In [1]:

from datetime import datetime

print(f"Today is {datetime.today():%A}.")

Today is Sunday.


In [2]:

from collections import Counter, defaultdict # For counting token pairs.
import pandas as pd # For loading the dataset.

In [3]:
africa_galore = pd.read_json(
    "https://storage.googleapis.com/dm-educational/assets/ai_foundations/africa_galore.json"
)
dataset = africa_galore["description"].values
print(f"Dataset contains {dataset.shape[0]:,} paragraphs.")

Dataset contains 232 paragraphs.


In [4]:
EOW_SYMBOL = ""

def bpe_initialize(dataset: list[str]) -> tuple[list[list[str]], set[str]]:
    """Implements the initialization step of the byte pair encoding algorithm.

    Args:
      dataset: The corpus consisting of a list of raw paragraphs.

    Returns:
      corpus: A list containing every word in the corpus represented as a list
        of characters and the special end-of-word-symbol "".
      vocabulary: The set of unique tokens in the dataset that serves as the
        vocabulary before the first merge.
    """

    corpus = []
    vocabulary = {EOW_SYMBOL}
    for paragraph in dataset:
        for word in paragraph.split(" "):
            # Convert word into chars and add end-of-word symbol ('').
            chars =  list(word)# Add your code here.
            corpus.append(chars)

            # Append characters to the vocabulary. Since vocabulary is a set,
            # it will ignore characters that have already been added.
            vocabulary.update(chars)

    return corpus, vocabulary

corpus, vocabulary = bpe_initialize(dataset)

# Print the first 10 "words" in the corpus.
print("First 10 words: ")
for word in corpus[:10]:
    print(f"  {word}")

print("\n\nVocabulary:")
print(f"  {vocabulary}")

First 10 words: 
  ['T', 'h', 'e']
  ['L', 'a', 'g', 'o', 's']
  ['a', 'i', 'r']
  ['w', 'a', 's']
  ['t', 'h', 'i', 'c', 'k']
  ['w', 'i', 't', 'h']
  ['h', 'u', 'm', 'i', 'd', 'i', 't', 'y', ',']
  ['b', 'u', 't']
  ['t', 'h', 'e']
  ['e', 'n', 'e', 'r', 'g', 'y']


Vocabulary:
  {'', 'a', 'H', '5', '2', 'j', 'I', '9', 'é', 'W', '—', 'z', 'o', ':', 'V', 's', 'y', 'N', 'R', 'P', ',', 'O', 'G', 'b', 'u', '”', 'i', 'l', 'x', ';', ')', '4', 'n', 'L', '"', 'A', 'h', '“', 'S', 'k', '0', '6', 'Y', 'c', "'", 'd', 'e', 'f', '8', '-', 'm', '/', 'w', '.', 't', 'M', 'v', '7', 'F', 'K', 'Z', 'C', 'T', 'p', 'E', '3', 'B', 'D', 'g', 'J', 'X', '°', '(', '1', 'q', 'U', 'r', '?'}


In [7]:
def get_pair_frequencies(
    corpus: list[list[str]]
) -> Counter[tuple[str, str], int]:
    """
    Calculates the frequency of adjacent character pairs in a corpus.

    Args:
      corpus: A list of tokenized words, where each word is represented as a
        list of subword tokens. Before the first merge these are all individual
        characters.

    Returns:
      A Counter object mapping each adjacent character pair (a tuple) to its
        frequency in the corpus.
    """
    pairs = Counter()
    for word in corpus:
        # Loop through the position of every token but the last one.
        for i in range(len(word) - 1):
            # Create a tuple representing the adjacent pair consisting of the
            # i-th and (i+1)-th token in `word`.
            pair = (word[i], word[i + 1]) # Add your code here.
            pairs[pair] += 1
    return pairs


pair_freqs = get_pair_frequencies(corpus)
print("The 10 most common pair frequencies are:")
for freq in pair_freqs.most_common(10):
    print(f" {freq}")

The 10 most common pair frequencies are:
 (('a', 'n'), 1883)
 (('t', 'h'), 1869)
 (('i', 'n'), 1822)
 (('h', 'e'), 1735)
 (('e', 'r'), 1359)
 (('n', 'd'), 1305)
 (('r', 'e'), 1185)
 (('e', 'd'), 1109)
 (('n', 'g'), 1083)
 (('e', 's'), 1057)


In [8]:
most_freq_pair, freq = pair_freqs.most_common(1)[0]
print(
    f"The most frequent pair is {most_freq_pair} with a frequency of {freq:,}."
)

new_token = most_freq_pair[0] + most_freq_pair[1]
vocabulary.add(new_token)


The most frequent pair is ('a', 'n') with a frequency of 1,883.


In [13]:
def merge_pair_in_word(
    word: list[str], pair_to_merge: tuple[str, str]
) -> list[str]:
    """Merges adjacent occurrences of a specified pair of characters in a word.

    Given a word represented as a list of subword tokens and a pair of tokens to
    merge (represented as a tuple), this function returns a new list where every
    instance of the specified pair appearing adjacently in the original word is
    replaced by a single string representing the merged pair.

    Args:
      tokens: A list of subword tokens representing one space separated word.
      pair_to_merge: A pair of two subword tokens that should be merged into one
        subword token.

    Returns:
      New list of subword tokens representing the word after applying the merge.
    """
    merged_symbol = pair_to_merge[0] + pair_to_merge[1]
    i = 0
    new_word = []
    while i < len(word):
        # If this position and the next match the pair, merge them.
        if i < len(word) - 1 and (word[i], word[i + 1]) == pair_to_merge: # Add your code here.
            new_word.append(merged_symbol)
            i += 2
        else:
            new_word.append(word[i])
            i += 1
    return new_word

print(f"Before merging: {' '.join(corpus[0])}")
new_word = merge_pair_in_word(corpus[0], most_freq_pair)
print(f"After merging:  {' '.join(new_word)}")


Before merging: T h e
After merging:  T h e


In [14]:

new_corpus = []
for word in corpus:
    new_word = merge_pair_in_word(word, most_freq_pair)
    new_corpus.append(new_word)

# Each iteration of BPE results in a new corpus with merged pairs.
print(
    "The first 10 \"words\" in the corpus, with \"e\" being a single token"
    " now:"
)
for word in new_corpus[:10]:
    print(f"  {word}")

print("\n\nVocabulary:")
print(f"  {vocabulary}")


The first 10 "words" in the corpus, with "e" being a single token now:
  ['T', 'h', 'e']
  ['L', 'a', 'g', 'o', 's']
  ['a', 'i', 'r']
  ['w', 'a', 's']
  ['t', 'h', 'i', 'c', 'k']
  ['w', 'i', 't', 'h']
  ['h', 'u', 'm', 'i', 'd', 'i', 't', 'y', ',']
  ['b', 'u', 't']
  ['t', 'h', 'e']
  ['e', 'n', 'e', 'r', 'g', 'y']


Vocabulary:
  {'', 'a', 'H', '5', '2', 0, 'j', 'I', '9', 'é', 'W', '—', 'z', 'o', ':', 'V', 's', 'y', 'N', 'R', 'P', 'an', ',', 'O', 'G', 'b', 'u', '”', 'i', 'l', 'x', ';', ')', '4', 'n', 'L', '"', 'A', 'h', '“', 'S', 'k', '0', '6', 'Y', 'c', "'", 'd', 'e', 'f', '8', '-', 'm', '/', 'w', '.', 't', 'M', 'v', '7', 'F', 'K', 'Z', 'C', 'T', 'p', 'E', '3', 'B', 'D', 'g', 'J', 'X', '°', '(', '1', 'q', 'U', 'r', '?'}


In [15]:

def learn_bpe(
    dataset: list[str], num_merges: int
) -> tuple[list[tuple[str, str]], list[list[str]]]:
    """
    Learns byte pair encoding (BPE) merge operations from a dataset.

    This function implements the basic BPE algorithm, iteratively merging the
    most frequent adjacent character pairs in a corpus to create subword units.
    It continues merging until the specified number of merges is reached or all
    words are represented by a single token.

    Args:
      dataset: A list of paragraphs, where each sentence is a string.
      num_merges: The desired number of BPE merge operations to perform.

    Returns:
      merges: A list of tuples, where each tuple represents a merged pair of
        tokens. (e.g., [('a', 'b'), ('ab', 'c')]).  The order of tuples reflects
        the order in which merges were performed.
      vocabulary: The final vocabulary of all subword tokens.
      corpus: The final corpus after BPE is applied.  Each sentence is
        represented as a list of words, and each word is represented as a list
        of subword tokens.
    """
    # Step 1: Initialize corpus as list of lists of characters + .
    corpus, vocabulary = bpe_initialize(dataset)

    merges = []
    for _ in range(num_merges):
        # Step 2: Count all adjacent-pair frequencies.
        pair_freqs = get_pair_frequencies(corpus)
        if not pair_freqs:
            break

        # Step 3: Pick the most frequent pair.
        most_freq_pair, freq = pair_freqs.most_common(1)[0]
        if freq < 1:
            break  # No more pairs to merge.
        merges.append(most_freq_pair)
        new_token = most_freq_pair[0] + most_freq_pair[1]
        vocabulary.add(new_token)

        # Step 4: Merge that pair in every word of the corpus.
        new_corpus = []
        for word in corpus:
            new_word = merge_pair_in_word(word, most_freq_pair)
            new_corpus.append(new_word)
        corpus = new_corpus

    # Return the list of merges, the vocabulary, and the final tokenized corpus.
    return merges, vocabulary, corpus

In [16]:

sample_text = [
        "desert",
        "deserted",
        "deserts",
        "desert",
        "tested",
        "test",
        "deserted",
        "desert",
        "desertion",
        "desertion",
        "function",
]
# Learn BPE tokenizer with 10 merge operations.
merges, vocabulary, tokenized_corpus = learn_bpe(sample_text, num_merges=10)

print("Learned merges (in order):")
for i, pair in enumerate(merges, start=1):
    print(f" {i}. Merge {pair[0]!r} + {pair[1]!r}  →  {pair[0]+pair[1]!r}")

print("\nFinal tokenized corpus:")
for orig, tokenized in zip(sample_text, tokenized_corpus):
    print(f"  {orig:8s} → {' '.join(tokenized)}")

Learned merges (in order):
 1. Merge 'e' + 's'  →  'es'
 2. Merge 'd' + 'es'  →  'des'
 3. Merge 'des' + 'e'  →  'dese'
 4. Merge 'dese' + 'r'  →  'deser'
 5. Merge 'deser' + 't'  →  'desert'
 6. Merge 'e' + 'd'  →  'ed'
 7. Merge 'i' + 'o'  →  'io'
 8. Merge 'io' + 'n'  →  'ion'
 9. Merge 'desert' + 'ed'  →  'deserted'
 10. Merge 't' + 'es'  →  'tes'

Final tokenized corpus:
  desert   → desert
  deserted → deserted
  deserts  → desert s
  desert   → desert
  tested   → tes t ed
  test     → tes t
  deserted → deserted
  desert   → desert
  desertion → desert ion
  desertion → desert ion
  function → f u n c t ion
